In [1]:
import os
import glob
import yaml
import numpy as np
import pandas as pd

The history saving thread hit an unexpected error (DatabaseError('database disk image is malformed')).History will not be written to the database.


# Prep dataset config

In [123]:
path_config = "/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/prep_dataset/K562_ATAC-seq.yaml"
path_out = "/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/prep_dataset"
os.makedirs(path_out, exist_ok=True)

In [124]:
config = yaml.safe_load(open(path_config, 'r'))
config

{'name': 'K562_ATAC-seq',
 'threads': 4,
 'random_state': 1234,
 'seqdata': {'fasta': '/cellar/users/aklie/data/ref/genomes/hg38/hg38.fa',
  'seq_var': 'seq',
  'bws': ['/cellar/users/aklie/data/datasets/SeqDatasets/K562_ATAC-seq/data/K562_ATAC-seq_unstranded_counts.bw'],
  'bw_names': ['SC.delta'],
  'cov_var': 'cov',
  'loci': '/cellar/users/aklie/data/datasets/SeqDatasets/K562_ATAC-seq/data/ENCSR868FGK_K562_ATAC-seq_peaks_no_blacklist.bed',
  'batch_size': 10000,
  'fixed_length': 2114,
  'target_length': 1000,
  'alphabet': 'DNA',
  'upper_case': False,
  'add_rev_comp': False,
  'max_jitter': 512},
 'negatives': {'gc_bin_width': 0.02,
  'max_n_perc': 0.1,
  'signal': '/cellar/users/aklie/data/datasets/SeqDatasets/K562_ATAC-seq/data/K562_ATAC-seq_unstranded_counts.bw',
  'signal_beta': 0.5,
  'in_window': 2114,
  'out_window': 1000,
  'random_state': 1234},
 'splits': '/cellar/users/aklie/projects/ML4GLand/tutorials/data/splits/ENCODE_cross-val.json'}

In [125]:
signal_betas = np.linspace(0.1, 1, 10)
signal_betas

array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1. ])

In [126]:
config_paths = []
for signal_beta in signal_betas:
    config['negatives']['signal_beta'] = round(float(signal_beta), 1)
    curr_out = os.path.join(path_out, str(round(signal_beta, 1)), "K562_ATAC-seq.yaml")
    os.makedirs(os.path.dirname(curr_out), exist_ok=True)
    with open(curr_out, 'w') as f:
        yaml.dump(config, f)
    print(curr_out)
    config_paths.append(curr_out)

/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/prep_dataset/0.1/K562_ATAC-seq.yaml
/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/prep_dataset/0.2/K562_ATAC-seq.yaml
/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/prep_dataset/0.3/K562_ATAC-seq.yaml
/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/prep_dataset/0.4/K562_ATAC-seq.yaml
/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/prep_dataset/0.5/K562_ATAC-seq.yaml
/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/prep_dataset/0.6/K562_ATAC-seq.yaml
/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/prep_dataset/0.7/K562_ATAC-seq.yaml
/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/prep_dataset/0.8/K562_ATAC-seq.yaml
/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/prep_dataset/0.9/K562_ATAC-seq.yaml
/cellar/us

In [127]:
# Save config_paths to .txt file in path_out
with open(os.path.join(path_out, "config_paths.txt"), 'w') as f:
    f.write("\n".join(config_paths))

In [128]:
# Make a dataframe with paths of configs in first column and output directory in second column
df = pd.DataFrame()
df["config"] = config_paths
df["out_dir"] = [os.path.dirname(p) for p in config_paths]
df.to_csv("/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/batch/metadata/prep_dataset.tsv", sep="\t", index=False, header=False)

# Bias fit configs

In [2]:
path_config = "/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/bias_model/K562_ATAC-seq_bias.yaml"
path_out = "/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/bias_model"

In [3]:
config = yaml.safe_load(open(path_config, 'r'))
config

{'name': 'K562_ATAC-seq_bias_fold_0',
 'threads': 4,
 'random_state': 1234,
 'seqdata': {'path': '/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/prep_dataset/0.5/K562_ATAC-seq.minimal.seqdata',
  'seq_var': 'seq',
  'cov_var': 'cov',
  'fold': 'fold_0',
  'seq_length': 2114,
  'target_length': 1000,
  'max_jitter': 0,
  'max_counts': 101,
  'min_counts': 0,
  'outlier_threshold': 0.9999},
 'model': {'n_filters': 128, 'n_layers': 4, 'n_outputs': 1, 'alpha': None},
 'training': {'learning_rate': 0.001,
  'batch_size': 64,
  'max_epochs': 50,
  'validation_iter': 1000,
  'rc_augment': True},
 'evaluation': {'batch_size': 256},
 'attribution': {'batch_size': 128, 'subsample': 30000, 'n_shuffles': 20},
 'modisco': {'n_seqlets': 50000,
  'window': 500,
  'motif_db': '/cellar/users/aklie/projects/ML4GLand/tutorials/data/motifs.meme.txt'}}

In [4]:
seqdatas = {}
for f in glob.glob(os.path.join(os.path.dirname(path_out), "prep_dataset", "*", "*.minimal.seqdata"), recursive=True):
    signal_beta = float(os.path.basename(os.path.dirname(f)))
    seqdatas[signal_beta] = f
seqdatas

{0.5: '/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/prep_dataset/0.5/K562_ATAC-seq.minimal.seqdata',
 0.8: '/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/prep_dataset/0.8/K562_ATAC-seq.minimal.seqdata',
 0.2: '/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/prep_dataset/0.2/K562_ATAC-seq.minimal.seqdata',
 0.6: '/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/prep_dataset/0.6/K562_ATAC-seq.minimal.seqdata',
 0.1: '/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/prep_dataset/0.1/K562_ATAC-seq.minimal.seqdata',
 0.3: '/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/prep_dataset/0.3/K562_ATAC-seq.minimal.seqdata',
 0.9: '/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/prep_dataset/0.9/K562_ATAC-seq.minimal.seqdata',
 0.4: '/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/prep_data

In [5]:
# Need to pull out Max Negative Counts: value from the following
reports = {}
for f in glob.glob(os.path.join(os.path.dirname(path_out), "prep_dataset", "*", "*.report.html"), recursive=True):
    signal_beta = float(os.path.basename(os.path.dirname(f)))
    with open(f, 'r') as f:
        content = f.read()
        max_neg_counts = float(content.split("Max Negative Counts: ")[1].split(",")[0])
        reports[signal_beta] = max_neg_counts
reports

{0.5: 103.0,
 0.8: 165.0,
 0.2: 41.0,
 0.6: 124.0,
 0.1: 20.0,
 0.3: 62.0,
 0.9: 186.0,
 0.4: 82.0,
 0.7: 144.0}

In [6]:
folds = 5

In [7]:
config_paths = []
for i, signal_beta in enumerate(seqdatas):
    for fold in range(folds):
        config["name"] = f"K562_ATAC-seq_bias_fold_{fold}" 
        config["seqdata"]["path"] = seqdatas[signal_beta]
        config["seqdata"]["fold"] = f"fold_{fold}"
        config["seqdata"]["max_counts"] = reports[signal_beta]
        curr_out = os.path.join(path_out, f"fold_{fold}", str(round(signal_beta, 1)), f"K562_ATAC-seq_bias_fold_{fold}.yaml")
        os.makedirs(os.path.dirname(curr_out), exist_ok=True)
        with open(curr_out, 'w') as f:
            yaml.dump(config, f)
        print(curr_out)
        config_paths.append(curr_out)

/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/bias_model/fold_0/0.5/K562_ATAC-seq_bias_fold_0.yaml
/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/bias_model/fold_1/0.5/K562_ATAC-seq_bias_fold_1.yaml
/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/bias_model/fold_2/0.5/K562_ATAC-seq_bias_fold_2.yaml


/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/bias_model/fold_3/0.5/K562_ATAC-seq_bias_fold_3.yaml
/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/bias_model/fold_4/0.5/K562_ATAC-seq_bias_fold_4.yaml
/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/bias_model/fold_0/0.8/K562_ATAC-seq_bias_fold_0.yaml
/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/bias_model/fold_1/0.8/K562_ATAC-seq_bias_fold_1.yaml
/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/bias_model/fold_2/0.8/K562_ATAC-seq_bias_fold_2.yaml
/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/bias_model/fold_3/0.8/K562_ATAC-seq_bias_fold_3.yaml
/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/bias_model/fold_4/0.8/K562_ATAC-seq_bias_fold_4.yaml
/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/bias_model/fold_0/0.2/K562_ATAC-seq_b

In [ ]:
# Keep only fold_0 configs
#config_paths = sorted([c for c in config_paths if "fold_0" in c])
#config_paths

In [8]:
# Keep only 0.5 signal_beta configs
config_paths = sorted([c for c in config_paths if "0.5" in c])
config_paths

['/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/bias_model/fold_0/0.5/K562_ATAC-seq_bias_fold_0.yaml',
 '/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/bias_model/fold_1/0.5/K562_ATAC-seq_bias_fold_1.yaml',
 '/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/bias_model/fold_2/0.5/K562_ATAC-seq_bias_fold_2.yaml',
 '/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/bias_model/fold_3/0.5/K562_ATAC-seq_bias_fold_3.yaml',
 '/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/bias_model/fold_4/0.5/K562_ATAC-seq_bias_fold_4.yaml']

In [9]:
# Make a dataframe with paths of configs in first column and output directory in second column
df = pd.DataFrame()
df["config"] = config_paths
df["out_dir"] = [os.path.dirname(p) for p in config_paths]
df.to_csv("/cellar/users/aklie/projects/ML4GLand/tutorials/bulk_atac_basepair/eugene/batch/metadata/bias_fit.tsv", sep="\t", index=False, header=False)

# ChromBPNet configs

# DONE!

---